In [ ]:
import sys
import os
sys.path.insert(0, '..')

from pathlib import Path
from privacyshield.text_pipeline.extractor import extract_text_pages
from privacyshield.pipeline import run_text_pipeline
from privacyshield.text_pipeline.pdf_rebuilder import rebuild_pdf_with_labels
from privacyshield.key_manager.encryptor import encrypt_token_map, key_to_string

# ---------------------------------------------------
# Redirect all output to a text file
# ---------------------------------------------------
log_file = open("redaction_results.txt", "w", encoding="utf-8")
sys.stdout = log_file

# ---------------------------------------------------
# Paths
# ---------------------------------------------------
TESTING_DIR = Path(r"C:\Users\Shrimi Agrawal\privacyshield\testing\GSF")
REDACTED_DIR = Path(r"C:\Users\Shrimi Agrawal\privacyshield\testing\redacted_output")
REDACTED_DIR.mkdir(parents=True, exist_ok=True)

print("Testing directory:", TESTING_DIR)
print("Exists:", TESTING_DIR.exists())

# ---------------------------------------------------
# Helper functions
# ---------------------------------------------------
def extract_pii_values(txt_path):
    pii_values = []
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" in line:
                _, value = line.split(":", 1)
                v = value.strip()
                if v:
                    pii_values.append(v)
    return pii_values


def check_redaction(pii_values, redacted_text):
    passed, failed = 0, 0
    for value in pii_values:
        if value in redacted_text:
            print(f"  ✗ NOT REDACTED: {value}")
            failed += 1
        else:
            print(f"  ✓ Redacted: {value}")
            passed += 1
    return passed, failed


# ---------------------------------------------------
# Main evaluation loop
# ---------------------------------------------------
total_passed = 0
total_failed = 0

for i in range(1, 101):

    pdf_path = TESTING_DIR / f"synthetic_doc_{i}.pdf"
    txt_path = TESTING_DIR / f"synthetic_pii_{i}.txt"

    out_pdf = REDACTED_DIR / f"synthetic_doc_redacted_{i}.pdf"
    shield_path = REDACTED_DIR / f"synthetic_pii_{i}.privacyshield"

    if not pdf_path.exists():
        print(f"[{i}] PDF NOT FOUND: {pdf_path}")
        continue

    if not txt_path.exists():
        print(f"[{i}] TXT NOT FOUND: {txt_path}")
        continue

    print(f"\n[{i}] {pdf_path.name}")

    try:
        result = run_text_pipeline(str(pdf_path))
        print(f"  Stats: {result['stats']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

    try:
        rebuild_pdf_with_labels(str(pdf_path), result, str(out_pdf), show_labels=True)
    except Exception as e:
        print(f"  ERROR saving PDF: {e}")
        continue

    try:
        key_bytes = encrypt_token_map(result["token_map"], str(shield_path))
        print(f"  Key: {key_to_string(key_bytes)[:20]}...")
    except Exception as e:
        print(f"  ERROR encrypting: {e}")

    try:
        extraction = extract_text_pages(str(out_pdf))
        redacted_text = "\n".join(p.full_text for p in extraction.pages)
    except Exception as e:
        print(f"  ERROR extracting redacted text: {e}")
        continue

    pii_values = extract_pii_values(txt_path)
    passed, failed = check_redaction(pii_values, redacted_text)

    total_passed += passed
    total_failed += failed


# ---------------------------------------------------
# Final summary
# ---------------------------------------------------
print("\n" + "=" * 50)
print(f"TOTAL: {total_passed} redacted ✓  |  {total_failed} missed ✗")

if total_passed + total_failed > 0:
    rate = 100 * total_passed / (total_passed + total_failed)
    print(f"Redaction rate: {rate:.1f}%")

# Close log file
log_file.close()